# Testing speed of data import

## Loading packages

In [1]:
using Revise
using Pkg;
#Pkg.develop(path="/home/gert/Projects/FusionRings.jl/")
#Pkg.develop(path="/Users/gertvercleyen/Projects/FusionRings.jl/")
using FusionRings
using Oscar
using JSON
using Base.Threads
using LinearAlgebra

[ Info: Precompiling FusionRings [609e252f-52eb-439d-abe9-2fbe746db898] (cache misses: incompatible header (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing FusionRings [609e252f-52eb-439d-abe9-2fbe746db898].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.



Welcome to Nemo version 0.52.4



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up



Nemo comes with absolutely no warranty whatsoever
 ┌───────┐   GAP 4.15.1 of 2025-10-18
 │  GAP  │   https://www.gap-system.org
 └───────┘   Architecture: x86_64-pc-linux-gnu-julia1.12-64-kv10
 Configuration:  gmp 6.3.0, Julia GC, Julia 1.12.3, readline
 Loading the library and packages ...
 Packages:   AClib 1.3.3, Alnuth 3.2.1, AtlasRep 2.1.9, AutoDoc 2025.10.16, 
             AutPGrp 1.11.1, Browse 1.8.21, CaratInterface 2.3.7, CRISP 1.4.8, 
             Cryst 4.1.30, CrystCat 1.1.10, CTblLib 1.3.11, 
             curlInterface 2.4.2, FactInt 1.6.3, FGA 1.5.0, Forms 1.2.13, 
             GAPDoc 1.6.7, genss 1.6.9, IO 4.9.3, IRREDSOL 1.4.4, 
             JuliaInterface 0.16.2, LAGUNA 3.9.7, orb 5.0.1, 
             PackageManager 1.6.3, Polenta 1.3.11, Polycyclic 2.17, 
             PrimGrp 4.0.1, RadiRoot 2.9, recog 1.4.4, ResClasses 4.7.4, 
             SmallGrp 1.5.4, Sophus 1.27, SpinSym 1.5.2, StandardFF 1.0, 
             TomLib 1.2.11, TransGrp 3.6.5, utils 0.92
 Try '??help'

## parallel_load

The data we will test on are
* The unique fpdims of all fusion rings, stored in "~/Tests/fpdims"
* The unique values of all characters of all fusion rings, stored in "~/tests/characters"

Each mrdi file contains a single qqbar number

In [7]:
base_dir = "/home/gert/Tests";
load_fp_dim(i::Int64) = Oscar.load( base_dir * "/fpdims/fpdim_" * string(i) * ".mrdi" );
ndims = 50903;

50903

In [5]:
function parallel_load( ) 

FR(2, 1, 0, 2)

In [4]:
dims = unique( vcat( [ fpdims(r) for r in frl ] ... ) );

In [8]:
@time [ load_fp_dim(i) for i in 1:ndims ]

 20.384064 seconds (37.39 M allocations: 2.403 GiB, 4.93% gc time, 18.59% compilation time)


50903-element Vector{QQBarFieldElem}:
 {a1: 1.00000}
 {a2: 1.61803}
 {a2: 1.41421}
 {a1: 2.00000}
 {a3: 1.80194}
 {a3: 2.24698}
 {a2: 2.41421}
 {a2: 2.61803}
 {a3: 1.87939}
 {a3: 2.53209}
 {a3: 2.87939}
 {a2: 1.73205}
 {a2: 2.30278}
 ⋮
 {a3: 17.5456}
 {a3: 24.1246}
 {a3: 8.29757}
 {a3: 19.1841}
 {a3: 17.4087}
 {a3: 21.7427}
 {a3: 16.6708}
 {a3: 24.8531}
 {a3: 6.34780}
 {a3: 18.2512}
 {a3: 9.46544}
 {a3: 20.3365}

In [12]:
@time begin 
    l = [ [], [], [], [] ]
    @threads for i in 1:length(dims)
        push!( l[Threads.threadid()-1], load_fp_dim(i) )
    end
    vcat( l... )
end

  5.021746 seconds (30.80 M allocations: 2.086 GiB, 16.49% gc time, 287 lock conflicts, 1.97% compilation time)


50903-element Vector{Any}:
 {a1: 1.00000}
 {a2: 1.61803}
 {a2: 1.41421}
 {a1: 2.00000}
 {a3: 1.80194}
 {a3: 2.24698}
 {a2: 2.41421}
 {a2: 2.61803}
 {a3: 1.87939}
 {a3: 2.53209}
 {a3: 2.87939}
 {a2: 1.73205}
 {a2: 2.30278}
 ⋮
 {a4: 28.5713}
 {a4: 9.91651}
 {a4: 23.8773}
 {a4: 24.5608}
 {a4: 14.7502}
 {a4: 19.7831}
 {a4: 20.5152}
 {a2: 33.6828}
 {a2: 34.6828}
 {a4: 17.5212}
 {a4: 22.5683}
 {a4: 25.3004}

In [18]:
function parallel_load( dirname::String )
    filenames = readdir( dirname )
    l = fill( [], Threads.nthreads() )
    @threads for fn in filenames
        push!( l[Threads.threadid()-1], Oscar.load(dirname * fn) )
    end
    vcat( l... )
end

parallel_load (generic function with 1 method)

In [19]:
@time chars = parallel_load( "/home/gert/Tests/characters/" );

 43.009472 seconds (237.02 M allocations: 15.184 GiB, 13.23% gc time, 6085 lock conflicts, 14.57% compilation time)


In [26]:
flat_chars = vcat( [ vcat( ch...) for ch in chars ]...)

3335952-element Vector{QQBarFieldElem}:
 {a1: 1.00000}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: 10.0990}
 {a2: -0.0990195}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: 11.0902}
 {a2: -0.0901699}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: 12.0828}
 {a2: -0.0827625}
 ⋮
 {a2: -0.500000 - 0.866025*im}
 {a2: -0.500000 + 0.866025*im}
 {a1: 1.00000}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: -0.500000 + 0.866025*im}
 {a2: -0.500000 - 0.866025*im}
 {a2: -0.500000 - 0.866025*im}
 {a2: -0.500000 + 0.866025*im}
 {a2: -0.500000 + 0.866025*im}
 {a2: -0.500000 - 0.866025*im}
 {a1: 1.00000}

In [27]:
unique_flat_chars = unique( flat_chars );

In [37]:
allnumbers = union( unique_flat_chars, vcat( l... ) );

In [42]:
unique(typeof.(allnumbers))

1-element Vector{DataType}:
 QQBarFieldElem

In [44]:
unique(degree.(minimal_polynomial.( allnumbers )))

9-element Vector{Int64}:
 1
 2
 3
 5
 6
 4
 7
 8
 9

In [45]:
length(allnumbers)

220027

In [51]:
replace( string(minpoly(allnumbers[234])),  "*" => "", " " => ""  )

"x^3-20x^2-26x+8"

In [103]:
function qqb_id( x::QQBarFieldElem ) 
    mp = minimal_polynomial(x)
    degstring = string( degree( mp ) )
    polstring = 
        replace( 
            string(mp),  
            "*" => "", " " => ""  
        )
    numstring = string( rootnum( x ) )

    degstring * "_" * polstring * "_" * numstring
    
end

function rootnum( x::QQBarFieldElem )
    p   = minimal_polynomial( x ) 
    rts = roots( QQBar, p )
    sr  = sort( rts, by = root_sort_crit )
    findfirst( y -> y == x, sr )
end

function root_sort_crit( x )
    ( - Int( is_real( x ) ), real(x), imag(x) )
end


function save_qqb_num( dir::String, x::QQBarFieldElem )
    idstring = qqb_id(x)
    Oscar.save( joinpath( dir, idstring * ".mrdi" ), idstring => x )
end

function save_qqb_num( x::QQBarFieldElem )
    save_qqb_num( joinpath(@__DIR__, "data","QQBarFieldElems"), x )
end

save_qqb_num (generic function with 2 methods)

In [102]:
save_qqb_num( "/home/gert/Projects/

"3_x^3-31x^2-122x-8_1"

In [89]:
x =  qqb( sqrt(qqb(1) + qqb(1im)) )
println( x )
println( rootnum( x ) )
roots( QQBar, minimal_polynomial( x ))

{a4: 1.09868 + 0.455090*im}
4


4-element Vector{QQBarFieldElem}:
 {a4: 1.09868 + 0.455090*im}
 {a4: 1.09868 - 0.455090*im}
 {a4: -1.09868 + 0.455090*im}
 {a4: -1.09868 - 0.455090*im}

In [96]:
?degree

search: degree degrees indegree is_degree degrevlex outdegree deglex wdegrevlex



```julia
degree(S::FunctionField)
```

Return the degree of the defining polynomial of the function field, i.e. the degree of the extension that the function field makes of the underlying rational function field.

---

```julia
degree(a::PolynomialElem)
```

Return the degree of the given polynomial. This is defined to be one less than the length, even for constant polynomials.

---

```julia
degree(a::MatRing)
```

Return the degree $n$ of the given matrix algebra.

---

```julia
degree(a::MatRingElem{T}) where T <: RingElement
```

Return the degree $n$ of the given matrix algebra.

---

```julia
degree(f::MPolyRingElem{T}, i::Int) where T <: RingElement
```

Return the degree of the polynomial $f$ in terms of the i-th variable.

---

```julia
degree(f::MPolyRingElem{T}, x::MPolyRingElem{T}) where T <: RingElement
```

Return the degree of the polynomial $f$ in terms of the variable $x$.

---

```julia
degree(R::GFField)
```

Return the degree of the given finite field.

---

```julia
degree(a::FqPolyRepField)
```

Return the degree of the given finite field.

---

```julia
degree(K::FqField) -> Int
```

Return the degree of the given finite field over the base field.

# Examples

```jldoctest
julia> K, a = finite_field(3, 2, "a");

julia> degree(K)
2

julia> Kx, x = K["x"];

julia> L, b = finite_field(x^3 + x^2 + x + 2, "b");

julia> degree(L)
3
```

---

```julia
degree(a::AbsSimpleNumField)
```

Return the degree of the given number field, i.e. the degree of its defining polynomial.

---

```julia
degree(x::QQBarFieldElem)
```

Return the degree of the minimal polynomial of `x`.

---

```julia
degree(L::NumField) -> Int
```

Given a number field $L/K$, this function returns the degree of $L$ over $K$.

# Examples

```jldoctest
julia> Qx, x = QQ["x"];

julia> K, a = number_field(x^2 - 2, "a");

julia> degree(K)
2
```

---

```julia
degree(P::AbsNumFieldOrderIdeal{AbsSimpleNumField, AbsSimpleNumFieldElem}) -> Int
```

The inertia degree of the prime-ideal $P$.

---

```julia
degree(O::NumFieldOrder) -> Int
```

Returns the degree of $\mathcal O$.

---

```julia
 degree(a::KInftyElem)
```

Return the degree of the given element, i.e. `degree(numerator) - degree(denominator)`.

---

```julia
degree(D::Divisor) -> Int
```

Return the degree of D.

---

```julia
degree(f::Isogeny) -> Int
```

Return the degree of the isogeny $f$.

---

```julia
degree(A::ClassField)
```

The degree of $A$ over its base field, i.e. the size of the defining ideal group.

---

```julia
degree(O::AlgAssAbsOrd) -> Int
```

Returns the dimension of the algebra containing $O$.

---

```julia
degree(O::AlgAssRelOrd) -> Int
```

Returns the dimension of the algebra containing $O$.

---

```julia
degree(L::AbstractLat) -> Int
```

Return the dimension of the ambient space of the lattice `L`.

---

```julia
degree(R::N_GField)
```

Return the degree of the field as an extension of $\mathbb{F}_p$.

---

```julia
degree(I::sideal{spoly{T}}) where T <: Nemo.RingElem
```

Return the (Krull) dimension and the multiplicity of the ideal generated by the leading monomials of the input. This is equal to the dimension and multiplicity of the ideal if the input is a standard basis with respect to a degree ordering.

---

```julia
degree(X::AbstractVariety)
```

If `X` has been given a polarization, return the corresponding degree of `X`.

# Examples

```jldoctest
julia> G = abstract_grassmannian(2,5)
AbstractVariety of dim 6

julia> degree(G)
5

```

!!! warning "Experimental"
    This function is part of the experimental code in Oscar. Please read [here](https://docs.oscar-system.org/v1/Experimental/intro/) for more details.


---

```julia
degree(f::MPolyDecRingElem)
```

Given a homogeneous element `f` of a graded multivariate ring, return the degree of `f`.

```julia
degree(::Type{Vector{Int}}, f::MPolyDecRingElem)
```

Given a homogeneous element `f` of a $\mathbb Z^m$-graded multivariate ring, return the degree of `f`, converted to a vector of integer numbers.

```julia
degree(::Type{Int}, f::MPolyDecRingElem)
```

Given a homogeneous element `f` of a $\mathbb Z$-graded multivariate ring, return the degree of `f`, converted to an integer number.

# Examples

```jldoctest
julia> G = abelian_group([0, 0, 2, 2])
Finitely generated abelian group
  with 4 generators and 4 relations and relation matrix
  [0   0   0   0]
  [0   0   0   0]
  [0   0   2   0]
  [0   0   0   2]

julia> W = [G[1]+G[3]+G[4], G[2]+G[4], G[1]+G[3], G[2], G[1]+G[2]];

julia> S, x = graded_polynomial_ring(QQ, :x => 1:5; weights = W)
(Graded multivariate polynomial ring in 5 variables over QQ, MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}[x[1], x[2], x[3], x[4], x[5]])

julia> f = x[2]^2+2*x[4]^2
x[2]^2 + 2*x[4]^2

julia> degree(f)
Abelian group element [0, 2, 0, 0]

julia> W = [[1, 0], [0, 1], [1, 0], [4, 1]]
4-element Vector{Vector{Int64}}:
 [1, 0]
 [0, 1]
 [1, 0]
 [4, 1]

julia> R, x = graded_polynomial_ring(QQ, :x => 1:4, W)
(Graded multivariate polynomial ring in 4 variables over QQ, MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}[x[1], x[2], x[3], x[4]])

julia> f = x[1]^4*x[2]+x[4]
x[1]^4*x[2] + x[4]

julia> degree(f)
[4 1]

julia> degree(Vector{Int}, f)
2-element Vector{Int64}:
 4
 1

julia>  R, (x, y, z) = graded_polynomial_ring(QQ, [:x, :y, :z], [1, 2, 3])
(Graded multivariate polynomial ring in 3 variables over QQ, MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}[x, y, z])

julia> f = x^6+y^3+z^2
x^6 + y^3 + z^2

julia> degree(f)
[6]

julia> typeof(degree(f))
FinGenAbGroupElem

julia> degree(Int, f)
6

julia> typeof(degree(Int, f))
Int64
```

---

```julia
degree(I::MPolyIdeal)
```

Given a (homogeneous) ideal `I` in a standard $\mathbb Z$-graded multivariate polynomial ring, return the degree of `I` (that is, the degree of the quotient of `base_ring(I)` modulo `I`). Otherwise, return the degree of the homogenization of `I` with respect to the standard $\mathbb Z$-grading.

!!! note
    Geometrically, the degree of a homogeneous ideal as above is the number of intersection points of its projective variety with a generic linear subspace of complementary dimension (counted with multiplicities). See also [MS21](@cite).


# Examples

```jldoctest
julia> R, (x, y, z) = polynomial_ring(QQ, [:x, :y, :z])
(Multivariate polynomial ring in 3 variables over QQ, QQMPolyRingElem[x, y, z])

julia> I = ideal(R, [y-x^2, x-z^3])
Ideal generated by
  -x^2 + y
  x - z^3

julia> degree(I)
6
```

---

```julia
degree(f::MPolyQuoRingElem{<:MPolyDecRingElem})
```

Given a homogeneous element `f` of a graded affine algebra, return the degree of `f`.

```julia
degree(::Type{Vector{Int}}, f::MPolyQuoRingElem{<:MPolyDecRingElem})
```

Given a homogeneous element `f` of a $\mathbb Z^m$-graded affine algebra, return the degree of `f`, converted to a vector of integer numbers.

```julia
degree(::Type{Int}, f::MPolyQuoRingElem{<:MPolyDecRingElem})
```

Given a homogeneous element `f` of a $\mathbb Z$-graded affine algebra, return the degree of `f`, converted to an integer number.

# Examples

```jldoctest
julia> R, (x, y, z) = graded_polynomial_ring(QQ, [:x, :y, :z]);

julia> A, p = quo(R, ideal(R, [y-x, z^3-x^3]))
(Quotient of multivariate polynomial ring by ideal (-x + y, -x^3 + z^3), Map: R -> A)

julia> f = p(y^2-x^2+z^4)
-x^2 + y^2 + z^4

julia> degree(f)
[4]

julia> typeof(degree(f))
FinGenAbGroupElem

julia> degree(Int, f)
4

julia> typeof(degree(Int, f))
Int64
```

---

```julia
degree(A::MPolyQuoRing)
```

Given a $\mathbb Z$-graded affine algebra $A = R/I$ over a field $K$, where the grading is inherited from the standard $\mathbb Z$-grading on the polynomial ring $R$, return the degree of $A$.

# Examples

```jldoctest
julia> R, (w, x, y, z) = graded_polynomial_ring(QQ, [:w, :x, :y, :z]);

julia> A, _ = quo(R, ideal(R, [w*y-x^2, w*z-x*y, x*z-y^2]));

julia> degree(A)
3
```

---

```julia
degree(G::PermGroup) -> Int
```

Return the degree of `G` as a permutation group, that is, an integer `n` that is stored in `G`, with the following meaning.

  * `G` embeds into `symmetric_group(n)`.
  * Two permutation groups of different degrees are regarded as not equal, even if they contain the same permutations.
  * Subgroups constructed with `derived_subgroup`, `sylow_subgroup`, etc., get the same degree as the given group.
  * The range `1:degree(G)` is used as the default set of points on which `G` and its element acts.
  * One can use the syntax `G(H)` in order to get a group that consists of the same permutations as `H` but has the same degree as `G`, provided that the elements of `H` move only points up to `degree(G)`.

!!! note
    The degree of a group of permutations is not necessarily equal to the largest moved point of the group `G`. For example, the trivial subgroup of `symmetric_group(n)` has degree `n` even though it fixes `n`.


# Examples

```jldoctest
julia> s4 = symmetric_group(4);

julia> degree(s4)
4

julia> t4 = trivial_subgroup(symmetric_group(4))[1];

julia> degree(t4)
4

julia> t5 = trivial_subgroup(symmetric_group(5))[1];

julia> t4 == t5
false

julia> t4 == s4(t5)
true

julia> show(Vector(gen(symmetric_group(4), 2)))
[2, 1, 3, 4]
julia> show(Vector(gen(symmetric_group(5), 2)))
[2, 1, 3, 4, 5]
```

---

```julia
degree(g::PermGroupElem) -> Int
```

Return the degree of the parent of `g`. This value is always greater or equal `number_of_moved_points(g)`

---

```julia
degree(c::CycleType) -> Int
```

Return the degree of the permutations with cycle structure `c`.

# Examples

```jldoctest
julia> g = symmetric_group(3);

julia> all(x -> degree(cycle_structure(x)) == degree(g), gens(g))
true
```

---

```julia
degree(G::MatrixGroup)
```

Return the degree of `G`, i.e., the number of rows of its matrices.

# Examples

```jldoctest
julia> degree(GL(4, 2))
4
```

---

```julia
degree(::Type{T} = QQFieldElem, chi::GAPGroupClassFunction)
       where T <: Union{IntegerUnion, QQFieldElem, QQAbFieldElem}
```

Return `chi[1]`, as an instance of `T`.

---

```julia
degree(f::FreeModElem{T}; check::Bool=true) where {T<:AnyGradedRingElem}
```

Given a homogeneous element `f` of a graded free module, return the degree of `f`.

```julia
degree(::Type{Vector{Int}}, f::FreeModElem)
```

Given a homogeneous element `f` of a $\mathbb Z^m$-graded free module, return the degree of `f`, converted to a vector of integer numbers.

```julia
degree(::Type{Int}, f::FreeModElem)
```

Given a homogeneous element `f` of a $\mathbb Z$-graded free module, return the degree of `f`, converted to an integer number.

If `check` is set to `false`, then there is no check for homegeneity. This should be called  internally on provably sane input, as it speeds up computation significantly. 

# Examples

```jldoctest
julia> R, (w, x, y, z) = graded_polynomial_ring(QQ, [:w, :x, :y, :z]);

julia> f = y^2*z − x^2*w
-w*x^2 + y^2*z

julia> degree(f)
[3]

julia> typeof(degree(f))
FinGenAbGroupElem

julia> degree(Int, f)
3

julia> typeof(degree(Int, f))
Int64
```

---

```julia
degree(a::FreeModuleHom; check::Bool=true)
```

If `a` is graded, return the degree of `a`.

# Examples

```jldoctest
julia> R, (x, y, z) = graded_polynomial_ring(QQ, [:x, :y, :z]);

julia> F = graded_free_module(R, 3)
Graded free module R^3([0]) of rank 3 over R

julia> G = graded_free_module(R, 2)
Graded free module R^2([0]) of rank 2 over R

julia> V = [y*G[1], x*G[1]+y*G[2], z*G[2]]
3-element Vector{FreeModElem{MPolyDecRingElem{QQFieldElem, QQMPolyRingElem}}}:
 y*e[1]
 x*e[1] + y*e[2]
 z*e[2]

julia> a = hom(F, G, V)
Graded module homomorphism of degree [1]
  from F
  to G
defined by
  e[1] -> y*e[1]
  e[2] -> x*e[1] + y*e[2]
  e[3] -> z*e[2]

julia> degree(a)
[1]
```

---

```julia
degree(m::SubquoModuleElem; check::Bool=true)
```

Given a homogeneous element `m` of a graded subquotient, return the degree of `m`.

```julia
degree(::Type{Vector{Int}}, m::SubquoModuleElem)
```

Given a homogeneous element `m` of a $\mathbb Z^m$-graded subquotient, return the degree of `m`, converted to a vector of integer numbers.

```julia
degree(::Type{Int}, m::SubquoModuleElem)
```

Given a homogeneous element `m` of a $\mathbb Z$-graded subquotient, return the degree of `m`, converted to an integer number.

# Examples

```jldoctest
julia> Rg, (x, y, z) = graded_polynomial_ring(QQ, [:x, :y, :z]);

julia> F1 = graded_free_module(Rg, [2,2,2]);

julia> F2 = graded_free_module(Rg, [2]);

julia> G = graded_free_module(Rg, [1,1]);

julia> V1 = [y*G[1], (x+y)*G[1]+y*G[2], z*G[2]];

julia> V2 = [z*G[2]+y*G[1]];

julia> a1 = hom(F1, G, V1);

julia> a2 = hom(F2, G, V2);

julia> M = subquotient(a1,a2);

julia> m = x*y*z*M[1]
x*y^2*z*e[1]

julia> degree(m)
[5]

julia> degree(Int, m)
5

julia> m3 = x*M[1]+M[2]+x*M[3]
(x*y + x + y)*e[1] + (x*z + y)*e[2]

julia> degree(m3)
[2]
```

---

```julia
degree(a::SubQuoHom; check::Bool=true)
```

If `a` is graded, return the degree of `a`.

# Examples

```jldoctest
julia> Rg, (x, y, z) = graded_polynomial_ring(QQ, [:x, :y, :z]);

julia> F = graded_free_module(Rg, 1);

julia> A = Rg[x; y];

julia> B = Rg[x^2; y^3; z^4];

julia> M = subquotient(F, A, B);

julia> N = M;

julia> V = [y^2*N[1], x^2*N[2]];

julia> a = hom(M, N, V)
Graded module homomorphism of degree [2]
  from M
  to M
defined by
  x*e[1] -> x*y^2*e[1]
  y*e[1] -> x^2*y*e[1]

julia> degree(a)
[2]
```

---

```julia
degree(a::FreeModElem_dec)
```

Return the degree of `a`. If `a` has no degree an error is thrown.

---

```julia
degree(Lf::ZZLatWithIsom) -> Int
```

Given a lattice with isometry $(L, f)$, return the degree of the underlying lattice $L$.

See [`degree(::AbstractLat)`](@ref).

# Examples

```jldoctest
julia> L = root_lattice(:A, 5);

julia> Lf = integer_lattice_with_isometry(L);

julia> degree(Lf)
5
```

---

```julia
degree(g::Graph{T} [, v::Int64]) where {T <: Union{Directed, Undirected}}
```

Return the degree of the vertex `v` in the graph `g`. If `v` is missing, return the list of degrees of all vertices. If the graph is directed, only neighbors reachable via outgoing edges are counted.

See also [`indegree`](@ref) and [`outdegree`](@ref) for directed graphs.

# Examples

```jldoctest
julia> g = vertex_edge_graph(icosahedron());

julia> degree(g, 1)
5
```

---

```julia
degree(l::ToricLineBundle)
```

Return the degree of the toric line bundle `l`.

# Examples

```jldoctest
julia> v = projective_space(NormalToricVariety, 2)
Normal toric variety

julia> l = toric_line_bundle(v, [ZZRingElem(2)])
Toric line bundle on a normal toric variety

julia> degree(l)
2
```

---

```julia
degree(C::ProjectivePlaneCurve)
```

Return the degree of the defining polynomial of `C`.

---

```julia
degree(p::ActionPolyRingElem, i::Int, jet::Vector{Int}) -> Int
```

Return the degree of the polynomial `p` in the jet variable specified by `i` and `jet`. If this jet variable is valid but still untracked, return $0$. This method allows all versions described in [Specifying jet variables](@ref specifying_jet_variables).

!!! warning "Experimental"
    This function is part of the experimental code in Oscar. Please read [here](https://docs.oscar-system.org/v1/Experimental/intro/) for more details.


---

```julia
degree(p::ActionPolyRingElem, i::Int)
```

Return the degree of the polynomial `p` in the, among the currently tracked jet variables, 'i'-th largest one. The index of the jet variable may also be passed as a tuple. Alternatively, the jet variable may be passed right away instead of its index.

!!! warning "Experimental"
    This function is part of the experimental code in Oscar. Please read [here](https://docs.oscar-system.org/v1/Experimental/intro/) for more details.

